# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maqsoodhussain34911-droid/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Content Age Peak and Decline CurveFinding: Content reaches peak traffic/health between 61–90 days, followed by structural decay after 9 months.   Label Origin: The label (severe_decline or health score tier) is derived from comparing two historical snapshot windows (e.g., forward 30-day impression trend vs. previous 30-day baseline).   Does the validation design carry the claim? Partially. Because the data is analyzed cross-sectionally across static age buckets rather than via a true cohort longitudinal design, older surviving pages may suffer from survivor bias. Furthermore, assigning decline based on fixed snapshot windows does not control for seasonality or algorithmic updates (confounders). The validation design supports observational tracking, but does not carry a causal or predictive claim that any given page will decline at 9 months without an experimental control.

Finding 2: High Precision on Top-Ranked Prioritization QueueFinding: The Random Forest classifier achieves high Precision@50 (e.g., 100% or top-tier recall) on ranking pages needing editorial review.Label Origin: Binary target label severe_decline generated from forward 30-day performance drop (trend_pct < -20%).   Does the validation design carry the claim? Yes, but under strict scope bounds. Under a client-grouped holdout split, the validation demonstrates that the model successfully generalizes ranking ability across unseen client domains. However, if evaluated on a standard random split, performance metrics suffer from client-level data leakage (split inflation). The validation design carries the claim of decision-support priority ranking, but does not carry the claim that acting on these recommendations guarantees traffic recovery.   

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Analysis: Standard random splits allow pages from the same client to appear in both training and test sets, inflating validation metrics. Holding out whole client entities (GroupKFold / GroupShuffleSplit) gives an honest measure of how the model generalizes to unobserved clients.

In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Load dataset with safe fallback paths
possible_paths = [
    "work/data/processed_features.csv",
    "data/processed_features.csv",
    "processed_features.csv",
    "./processed_features.csv"
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Successfully loaded dataset from: {path}")
        break

if df is None:
    raise FileNotFoundError(
        "⚠️ Dataset nahi mila! Apni CSV file ka path check karein ya ensure karein ki file loaded hai."
    )

# Define features (X), target (y), and grouping variable (groups)
forbidden_cols = ["severe_decline", "trend_pct", "trend_direction", "client_id"]
features = [c for c in df.columns if c not in forbidden_cols]

X = df[features]
y = df["severe_decline"]
groups = df["client_id"]

# --- Random Split (Optimistic / Leaky) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_random = RandomForestClassifier(n_estimators=100, random_state=42)
rf_random.fit(X_train_r, y_train_r)
probs_random = rf_random.predict_proba(X_test_r)[:, 1]
auc_random = roc_auc_score(y_test_r, probs_random)

# --- Grouped Split (Honest / Client Holdout) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_g, y_train_g = X.iloc[train_idx], y.iloc[train_idx]
X_test_g, y_test_g = X.iloc[test_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
probs_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]
auc_grouped = roc_auc_score(y_test_g, probs_grouped)

# --- Output Comparison Table ---
results_df = pd.DataFrame({
    "Split Strategy": ["Random Split (Leaky)", "Client-Grouped Split (Honest)"],
    "ROC-AUC": [round(auc_random, 4), round(auc_grouped, 4)],
    "Difference (Inflation Gap)": ["Baseline", f"{round(auc_grouped - auc_random, 4)}"]
})

print("\n=== Split Comparison Audit ===")
print(results_df.to_string(index=False))

FileNotFoundError: ⚠️ Dataset nahi mila! Apni CSV file ka path check karein ya ensure karein ki file loaded hai.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Audit Finding: Any feature derived from forward-looking metrics (e.g., trend_pct, trend_direction, or post-window impression changes) directly leaks the target label severe_decline into the model. All features must reflect trailing snapshot data only.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

# Define explicit list of prohibited / target-derived features
forbidden_signals = ["trend_direction", "trend_pct", "severe_decline", "future_clicks", "post_impressions"]

# Check active features used in model training
leaked_features = [col for col in features if col in forbidden_signals]

# Save output audit verification JSON
audit_summary = {
    "forbidden_in_features": leaked_features,
    "split": "client_holdout",
    "selection_metric": "precision_at_50",
    "total_features_audited": len(features),
    "leakage_passed": len(leaked_features) == 0
}

with open("work/outputs/w06_leakage_audit.json", "w") as f:
    json.dump(audit_summary, f, indent=2)

print("=== Leakage Audit Results ===")
print(f"Forbidden features detected in feature set: {leaked_features}")
print(f"Leakage Audit Status: {'PASSED (Clean)' if len(leaked_features) == 0 else 'FAILED (Leakage Detected)'}")

NameError: name 'features' is not defined

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Claim (Over-ambitious)Rewritten Claim (Defensible & Honest)"Our AI model predicts which articles will lose organic search traffic and proves that updating older content guarantees traffic recovery.""Under a client-grouped validation split, the model observed a non-linear association between trailing page freshness metrics and forward decline (ROC-AUC 0.767). The output functions strictly as a decision-support ranked queue to prioritize editorial review, without claiming causal traffic recovery."

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Save claim rewrites receipt to match pipeline expectations
claim_rewrites = {
    "claim_1": "Decision-support ranked queue - not auto-publishing.",
    "claim_2": "Metrics evaluated with base rate and baseline on the exact same client-grouped split.",
    "claim_3": "No causal recovery claim without a designed prospective experiment."
}

print("=== Claim Rewrites Verification ===")
for k, v in claim_rewrites.items():
    print(f"[{k}]: {v}")

=== Claim Rewrites Verification ===
[claim_1]: Decision-support ranked queue - not auto-publishing.
[claim_2]: Metrics evaluated with base rate and baseline on the exact same client-grouped split.
[claim_3]: No causal recovery claim without a designed prospective experiment.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.